# Mini-MLP Design/Build Assignment
## Hannah Zeilstra September 2026

## Imports

In [ ]:
import random
import numpy as np

## 1. Create Data
Each row of `X` is one sample with 3 binary inputs, and `y` holds the target output for each row.

The target is **3-bit parity**: `y = 1` when the row contains an even number of 1s. Flipping any single input flips the answer, so no single straight line can separate the two classes. That makes this a hard pattern for a small network.

In [25]:
# 8 samples x 3 binary inputs
X=np.array([[0,0,1],[0,1,1],[1,0,1],[1,1,1],[0,0,0],[0,1,0],[1,0,0],[1,1,0]],dtype=float)
# target output for each sample (3-bit parity)
y=np.array([0,1,1,0,1,0,0,1])

## 2. Helper Functions
- `sigmoid`: activation function that maps any value to a number between 0 and 1
- `generate_weights`: creates `n` random starting weights between -1 and 1
- `perceptron_pass`: runs every sample through one perceptron (weighted sum, then sigmoid)
- `calculate_accuracy`: rounds the outputs at 0.5 and returns the fraction of predictions that match `y`

In [26]:
# activation function: squashes any value into the range (0, 1)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# create n random weights between -1 and 1
def generate_weights(n):
    weights=[]
    for _ in range(n):
        weights.append(random.uniform(-1,1))
    return weights

# one perceptron: weighted sum of the inputs, then sigmoid
# x is (samples x inputs), so this returns one output per sample
def perceptron_pass(x, weights):
    outputs=(sigmoid(x @ weights))
    return outputs

# fraction of samples classified correctly (output >= 0.5 counts as 1)
def calculate_accuracy(outputs, y):
    return((outputs >= 0.5).astype(int)==y).mean()



## 3. Train the Networks (Random Hill-Climbing)
For each run, and for each network size:
1. Start with random weights. The 2-hidden network needs 3×2 + 2 = **8 weights**; the 3-hidden network needs 3×3 + 3 = **12 weights**.
2. Copy the best weights so far and change one randomly chosen weight by a random amount between -1 and +1.
3. Run the data forward through the network and calculate accuracy.
4. If accuracy **improved**, keep the new weights and reset the failure counter. Otherwise, throw the change away (the next attempt starts from the best weights again) and add 1 to the failure counter.
5. Once there have been 1,000,000 failures in a row, write the best accuracy and weights to a file.

**Weight layout** (the 2-hidden network is the same idea, with 2 blocks of 3):

| Weights | Connects |
|---|---|
| `w[0:3]` | inputs → hidden perceptron 1 |
| `w[3:6]` | inputs → hidden perceptron 2 |
| `w[6:9]` | inputs → hidden perceptron 3 |
| `w[9:12]` | hidden perceptrons → output |

Seeding with the run number makes every run reproducible while still giving each run different starting weights.

In [27]:
# 3 independent runs; each writes one 2-hidden file and one 3-hidden file
for run in range(1, 4):  
    random.seed(run)        # seed with the run number so results are reproducible
    # ---- 2 hidden ----
    # 8 weights: 3 inputs x 2 hidden + 2 hidden -> output
    w=generate_weights(8)
    best_weights, best_acc=w,0
    failure=0
    # stop after 1,000,000 attempts in a row with no improvement
    while failure < 1000:

        # start from the best weights and nudge one random weight up or down
        w=best_weights.copy()
        w[random.randint(0,len(w)-1)]+=random.uniform(-1,1)

        # hidden layer: each perceptron gets all 3 inputs
        p1=perceptron_pass(X, w[0:3])
        p2=perceptron_pass(X, w[3:6])

        # output layer: hidden outputs become the inputs (8 samples x 2)
        out_p=perceptron_pass(np.column_stack([p1,p2]), w[6:8])
        a=calculate_accuracy(out_p,y)
    
        # keep the change only if accuracy improved; otherwise it's discarded
        # because the next attempt starts again from best_weights
        if a > best_acc:
            best_acc=a
            best_weights=w
            failure=0
        else: 
            failure +=1

    # save the best result for this run
    with open(f"run{run}_2_hidden.txt", "w") as f:
        f.write(f"Run {run} - 2 hidden perceptrons\n"
                f"Accuracy: {best_acc}\nWeights: {best_weights}\n")

    # ---- 3 hidden ----

    # 12 weights: 3 inputs x 3 hidden + 3 hidden -> output
    w=generate_weights(12)
    best_weights, best_acc=w,0
    failure=0
    while failure < 1000:

        # start from the best weights and nudge one random weight up or down
        w=best_weights.copy()
        w[random.randint(0,len(w)-1)]+=random.uniform(-1,1)

        # hidden layer: 3 perceptrons, each with its own 3 weights
        p1=perceptron_pass(X, w[0:3])
        p2=perceptron_pass(X, w[3:6])
        p3=perceptron_pass(X, w[6:9])

        # output layer: 8 samples x 3 hidden outputs
        out_p=perceptron_pass(np.column_stack([p1,p2,p3]), w[9:12])
        a=calculate_accuracy(out_p,y)
    
        # keep if better, otherwise discard and count a failure
        if a > best_acc:
            best_acc=a
            best_weights=w
            failure=0
        else: 
            failure +=1

    # save the best result for this run
    with open(f"run{run}_3_hidden.txt", "w") as f:
        f.write(f"Run {run} - 3 hidden perceptrons\n"
                f"Accuracy: {best_acc}\nWeights: {best_weights}\n")


## 4. Results

| Run (seed) | 2 hidden | 3 hidden |
|---|---|---|
| 1 | 0.75 | 0.75 |
| 2 | 0.625 | 0.75 |
| 3 | 0.75 | 0.50 |
| **Average** | **0.708** | **0.667** |

Adding a third hidden perceptron did not improve the results. Every run got stuck at a local optimum: even after 1,000,000 attempts in a row, no single-weight change could raise the accuracy. Because there are only 8 samples, accuracy changes in steps of 1/8, and most small weight changes don't flip any prediction, so the search usually sees no difference to follow.

## 5. AI Disclosure
I used Claude to help me format the notebook and generate markdowns and comments for my code. I think its explanations are more thorough than my own and helps me for future reference.